# __Jupyter Notebook: DATA 22700 Final Project__
## Following deportation trends in the US for the last ten years
### (2015 - 2025)

This project is intended to follow trends in deportation data from two sources: the U.S. Immigration and Customs Enforcement statistics site and the Deportation Data site on historical deportation data.

### How has ICE detention changed over time?
#### What this can reveal: variables that are tied into certain changes in ICE detention and deportation rates

narrative hook: ...

In [4]:
import pandas as pd
import altair as alt
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [5]:
ice_arrests = pd.read_excel("./streamlit_fodler/data/ICE_data.xlsx")
ice_arrests.head()

,Criminality,Area of Responsibility (AOR),Country of Citizenship,Fiscal Year,Fiscal Quarter,Fiscal Month,Month-Year,Administrative Arrests
0,Criminal Conviction,Atlanta,COLOMBIA,2022,2,4,Jan 2022,10
1,Criminal Conviction,Atlanta,COLOMBIA,2022,3,8,May 2022,10
2,Criminal Conviction,Atlanta,COLOMBIA,2023,1,1,Oct 2022,12
3,Criminal Conviction,Atlanta,COLOMBIA,2023,2,5,Feb 2023,18
4,Criminal Conviction,Atlanta,COLOMBIA,2024,1,1,Oct 2023,16


In [7]:
ice_arrests.info()

<class 'pandas.DataFrame'>
RangeIndex: 9707 entries, 0 to 9706
Data columns (total 8 columns):
 #   Column                        Non-Null Count  Dtype
---  ------                        --------------  -----
 0   Criminality                   9707 non-null   str  
 1   Area of Responsibility (AOR)  9707 non-null   str  
 2   Country of Citizenship        9707 non-null   str  
 3   Fiscal Year                   9707 non-null   int64
 4   Fiscal Quarter                9707 non-null   int64
 5   Fiscal Month                  9707 non-null   int64
 6   Month-Year                    9707 non-null   str  
 7   Administrative Arrests        9707 non-null   int64
dtypes: int64(4), str(4)
memory usage: 606.8 KB


Provenance: U.S. Immigrations and Customs Enforcement
Link: https://www.ice.gov/statistics

Within the ICE Excel data sheet, there are many pages...

In [11]:
ice_atd = pd.read_excel("./streamlit_fodler/data/ICE_data.xlsx", sheet_name="ICE ATD")
ice_arrests = pd.read_excel("./streamlit_fodler/data/ICE_data.xlsx", sheet_name="ICE-ERO Administrative Arrests")
ice_detentions = pd.read_excel("./streamlit_fodler/data/ICE_data.xlsx", sheet_name="ICE Detentions")
ice_removals = pd.read_excel("./streamlit_fodler/data/ICE_data.xlsx", sheet_name="ICE Removals")
ice_ex_individuals = pd.read_excel("./streamlit_fodler/data/ICE_data.xlsx", sheet_name='ICE T42 Expulsions Indivduals')
ice_ex_flights = pd.read_excel("./streamlit_fodler/data/ICE_data.xlsx", sheet_name='ICE T42 Expulsions Flights ')

## ICE Data
The ICE Data contains five relevant tables:

ICE Alternatives to Detention Data: Featuring numbers of migrants who are not physically detained but are monitored using other methods.
ICE Arrests Data: Featuring the number of individuals arrested by ICE. Different from detentions, arrests mean an officer apprehended someone.
ICE Detentions Data: Featuring the number of individuals detained by ICE. Different from arrests, detentions mean an individual was held in custody at a detention center.
ICE Removals Data: Featuring numbers of individuals that were deported from the U.S.
ICE T42 Expulsions Individual Data: Features the number of expulsion of individuals under the Title 42 (T42) order, which allows the expulsion of individuals from the country without normal immigration processing.
ICE T42 Expulsions Flight Data: Features the number of T42 Expulsion related flights.

Now, to visualize yearly ICE activity, we can combine the tables of ICE data into one 

In [12]:
atd_yearly = ice_atd.groupby("Fiscal Year").size().reset_index(name="ATD")
arrests_yearly = ice_arrests.groupby("Fiscal Year").size().reset_index(name="Arrests")
detentions_yearly = ice_detentions.groupby("Fiscal Year").size().reset_index(name="Detentions")
removals_yearly = ice_removals.groupby("Fiscal Year").size().reset_index(name="Removals")
expulsions_yearly = ice_ex_individuals.groupby("Fiscal Year").size().reset_index(name="T42 Expulsions")

combined = atd_yearly.merge(arrests_yearly, on="Fiscal Year", how="outer") \
    .merge(detentions_yearly, on="Fiscal Year", how="outer") \
    .merge(removals_yearly, on="Fiscal Year", how="outer") \
    .merge(expulsions_yearly, on="Fiscal Year", how="outer")
combined = combined.fillna(0)

combined["Total"] = (
    combined["ATD"] +
    combined["Arrests"] +
    combined["Detentions"] +
    combined["Removals"] +
    combined["T42 Expulsions"]
)

long_df = combined.melt(
    id_vars="Fiscal Year",
    var_name="Enforcement_Type",
    value_name="Count"
)

selector = alt.selection_point(
    fields=["Enforcement_Type"],
    bind=alt.binding_select(
        options=[
            "ATD",
            "Arrests",
            "Detentions",
            "Removals",
            "T42 Expulsions",
            "Total"
        ],
        name="Enforcement Type: "
    ),
)

In [13]:
chart = (
    alt.Chart(long_df)
    .mark_bar()
    .encode(
        x=alt.X("Fiscal Year:O", title="Fiscal Year"),
        y=alt.Y("Count:Q", title="Count"),
        color="Fiscal Year:O",
        tooltip=["Fiscal Year", "Enforcement_Type", "Count"]
    )
    .add_params(selector)
    .transform_filter(selector)
    .properties(
        title="ICE Enforcement Actions by Fiscal Year"
    )
)

In [14]:
chart

alt.Chart(...)